In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


 # STAGE 2A-1 — ResNet-50 baseline
# Fixed stratified split | Standard augmentation

In [ ]:
import os
import random
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)


# 1. Reproducibility


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 2. Load the fixed split


SPLIT_FILE = Path(
    "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/"
    "betel_leaf_stratified_split.csv"
)

df = pd.read_csv(SPLIT_FILE)

print("\nSplit file loaded:")
print(SPLIT_FILE)

print("\nTotal images:", len(df))

print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nClass distribution:")
print(pd.crosstab(df["class"], df["split"]))


# 3. Class labels


classes = sorted(df["class"].unique())

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(classes)
}

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}

print("\nClasses:")
for idx, class_name in idx_to_class.items():
    print(idx, "=", class_name)


# 4. Dataset


class BetelLeafDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        label = class_to_idx[row["class"]]

        if self.transform:
            image = self.transform(image)

        return image, label



# 5. Standard augmentation



IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# 6. Create datasets


train_data = df[df["split"] == "train"]
val_data = df[df["split"] == "validation"]
test_data = df[df["split"] == "test"]

train_dataset = BetelLeafDataset(
    train_data,
    transform=train_transform
)

val_dataset = BetelLeafDataset(
    val_data,
    transform=eval_transform
)

test_dataset = BetelLeafDataset(
    test_data,
    transform=eval_transform
)


# 7. Data loaders


BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


# 8. ResNet-50


weights = models.ResNet50_Weights.DEFAULT

model = models.resnet50(
    weights=weights
)

# Replace final classification layer
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(classes)
)

model = model.to(DEVICE)

print("\nModel:")
print("ResNet-50")
print("Parameters:",
      sum(p.numel() for p in model.parameters()))


# 9. Loss, optimizer, scheduler


criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


# 10. Training configuration


NUM_EPOCHS = 15

best_val_f1 = -1
best_epoch = 0
best_state = None

history = []

start_time = time.time()


# 11. Training loop


for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()


    # Training


    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total


    # Validation


    model.eval()

    val_loss = 0.0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= len(val_dataset)

    val_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision, recall, val_f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    scheduler.step(val_f1)


    # Save best model


    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1

        best_state = copy.deepcopy(
            model.state_dict()
        )

    epoch_time = time.time() - epoch_start

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_f1,
        "epoch_time_sec": epoch_time
    })

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )


# 12. Restore best validation model


model.load_state_dict(best_state)

total_training_time = time.time() - start_time

print("\n")
print("RESNET-50 BASELINE COMPLETE")


print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", round(best_val_f1, 4))
print("Training time (minutes):",
      round(total_training_time / 60, 2))


# 13. Detailed validation report


model.eval()

val_labels = []
val_predictions = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(DEVICE)

        outputs = model(images)

        predictions = outputs.argmax(
            dim=1
        )

        val_labels.extend(
            labels.numpy()
        )

        val_predictions.extend(
            predictions.cpu().numpy()
        )

print("\nValidation classification report:")
print(
    classification_report(
        val_labels,
        val_predictions,
        target_names=classes,
        digits=4,
        zero_division=0
    )
)

print("Validation confusion matrix:")
print(
    confusion_matrix(
        val_labels,
        val_predictions
    )
)


# 14. Save training history


history_df = pd.DataFrame(history)

# Fixed directly instead of using undefined DATASET_DIR
RESULTS_DIR = Path(
    "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/"
    "Stage2A_Results"
)

RESULTS_DIR.mkdir(
    exist_ok=True
)

history_file = (
    RESULTS_DIR /
    "efficientnet_b0_history.csv"
)

history_df.to_csv(
    history_file,
    index=False
)


# 15. Save best model


model_file = (
    RESULTS_DIR /
    "resnet50_baseline_best.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "classes": classes,
        "class_to_idx": class_to_idx,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1,
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-4
    },
    model_file
)

print("\nSaved:")
print(history_file)
print(model_file)

PyTorch: 2.11.0+cu130
Device: cuda

Split file loaded:
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/betel_leaf_stratified_split.csv

Total images: 720

Split distribution:
split
train         504
test          108
validation    108
Name: count, dtype: int64

Class distribution:
split      test  train  validation
class                             
bacterial    37    171          37
fungal       34    160          34
healthy      37    173          37

Classes:
0 = bacterial
1 = fungal
2 = healthy

Dataset sizes:
Train: 504
Validation: 108
Test: 108
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 209MB/s]



Model:
ResNet-50
Parameters: 23514179
Epoch 01/15 | Train Loss: 0.9581 | Train Acc: 0.6687 | Val Loss: 0.7422 | Val Acc: 0.8704 | Val Macro-F1: 0.8685
Epoch 02/15 | Train Loss: 0.5347 | Train Acc: 0.8968 | Val Loss: 0.2357 | Val Acc: 0.8981 | Val Macro-F1: 0.8986
Epoch 03/15 | Train Loss: 0.2366 | Train Acc: 0.9345 | Val Loss: 0.1496 | Val Acc: 0.9537 | Val Macro-F1: 0.9546
Epoch 04/15 | Train Loss: 0.1186 | Train Acc: 0.9722 | Val Loss: 0.1061 | Val Acc: 0.9722 | Val Macro-F1: 0.9727
Epoch 05/15 | Train Loss: 0.0600 | Train Acc: 0.9881 | Val Loss: 0.1388 | Val Acc: 0.9537 | Val Macro-F1: 0.9537
Epoch 06/15 | Train Loss: 0.0479 | Train Acc: 0.9921 | Val Loss: 0.1189 | Val Acc: 0.9630 | Val Macro-F1: 0.9632
Epoch 07/15 | Train Loss: 0.0411 | Train Acc: 0.9901 | Val Loss: 0.1045 | Val Acc: 0.9722 | Val Macro-F1: 0.9727
Epoch 08/15 | Train Loss: 0.0246 | Train Acc: 0.9980 | Val Loss: 0.1013 | Val Acc: 0.9630 | Val Macro-F1: 0.9632
Epoch 09/15 | Train Loss: 0.0171 | Train Acc: 0.9980 | Va

In [ ]:
import os
import random
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

# 1. Reproducibility

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 2. Load the fixed split

SPLIT_FILE = Path(
    "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/"
    "betel_leaf_stratified_split.csv"
)

df = pd.read_csv(SPLIT_FILE)

print("\nSplit file loaded:")
print(SPLIT_FILE)

print("\nTotal images:", len(df))

print("\nSplit distribution:")
print(df["split"].value_counts())

print("\nClass distribution:")
print(pd.crosstab(df["class"], df["split"]))

# 3. Class labels

classes = sorted(df["class"].unique())

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(classes)
}

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}

print("\nClasses:")
for idx, class_name in idx_to_class.items():
    print(idx, "=", class_name)


# 4. Dataset

class BetelLeafDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        label = class_to_idx[row["class"]]

        if self.transform:
            image = self.transform(image)

        return image, label



# 5. Standard augmentation


IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# 6. Create datasets


train_data = df[df["split"] == "train"]
val_data = df[df["split"] == "validation"]
test_data = df[df["split"] == "test"]

train_dataset = BetelLeafDataset(
    train_data,
    transform=train_transform
)

val_dataset = BetelLeafDataset(
    val_data,
    transform=eval_transform
)

test_dataset = BetelLeafDataset(
    test_data,
    transform=eval_transform
)


# 7. Data loaders


BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


# 8. EfficientNet-B0


weights = models.EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(
    weights=weights
)

# Replace final classification layer
num_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(
    num_features,
    len(classes)
)

model = model.to(DEVICE)

print("\nModel:")
print("EfficientNet-B0")
print("Parameters:",
      sum(p.numel() for p in model.parameters()))


# 9. Loss, optimizer, scheduler


criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)


# 10. Training configuration


NUM_EPOCHS = 15

best_val_f1 = -1
best_epoch = 0
best_state = None

history = []

start_time = time.time()


# 11. Training loop


for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()

    # -------------------------
    # Training
    # -------------------------

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(
            dim=1
        )

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total


    # Validation


    model.eval()

    val_loss = 0.0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item()
                * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss /= len(val_dataset)

    val_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision, recall, val_f1, _ = (
        precision_recall_fscore_support(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )
    )

    scheduler.step(val_f1)


    # Save best model


    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_epoch = epoch + 1

        best_state = copy.deepcopy(
            model.state_dict()
        )

    epoch_time = time.time() - epoch_start

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_f1,
        "epoch_time_sec": epoch_time
    })

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )


# 12. Restore best validation model


model.load_state_dict(best_state)

total_training_time = time.time() - start_time

print("\n")
print("EFFICIENTNET-B0 BASELINE COMPLETE")


print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", round(best_val_f1, 4))
print("Training time (minutes):",
      round(total_training_time / 60, 2))


# 13. Detailed validation report


model.eval()

val_labels = []
val_predictions = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(DEVICE)

        outputs = model(images)

        predictions = outputs.argmax(
            dim=1
        )

        val_labels.extend(
            labels.numpy()
        )

        val_predictions.extend(
            predictions.cpu().numpy()
        )

print("\nValidation classification report:")
print(
    classification_report(
        val_labels,
        val_predictions,
        target_names=classes,
        digits=4,
        zero_division=0
    )
)

print("Validation confusion matrix:")
print(
    confusion_matrix(
        val_labels,
        val_predictions
    )
)


# 14. Save training history


history_df = pd.DataFrame(history)

# Fixed directly instead of using undefined DATASET_DIR
RESULTS_DIR = Path(
    "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/"
    "Stage2A_Results"
)

RESULTS_DIR.mkdir(
    exist_ok=True
)

history_file = (
    RESULTS_DIR /
    "efficientnet_b0_history.csv"
)

history_df.to_csv(
    history_file,
    index=False
)


# 15. Save best model


model_file = (
    RESULTS_DIR /
    "efficientnet_b0_best.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "classes": classes,
        "class_to_idx": class_to_idx,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1,
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-4
    },
    model_file
)

print("\nSaved:")
print(history_file)
print(model_file)

PyTorch: 2.11.0+cu130
Device: cuda

Split file loaded:
/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/betel_leaf_stratified_split.csv

Total images: 720

Split distribution:
split
train         504
test          108
validation    108
Name: count, dtype: int64

Class distribution:
split      test  train  validation
class                             
bacterial    37    171          37
fungal       34    160          34
healthy      37    173          37

Classes:
0 = bacterial
1 = fungal
2 = healthy

Dataset sizes:
Train: 504
Validation: 108
Test: 108
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 55.5MB/s]



Model:
EfficientNet-B0
Parameters: 4011391
Epoch 01/15 | Train Loss: 0.9663 | Train Acc: 0.5933 | Val Loss: 0.7672 | Val Acc: 0.8519 | Val Macro-F1: 0.8522
Epoch 02/15 | Train Loss: 0.6505 | Train Acc: 0.8313 | Val Loss: 0.4451 | Val Acc: 0.8796 | Val Macro-F1: 0.8810
Epoch 03/15 | Train Loss: 0.4115 | Train Acc: 0.8770 | Val Loss: 0.2834 | Val Acc: 0.8981 | Val Macro-F1: 0.8999
Epoch 04/15 | Train Loss: 0.2767 | Train Acc: 0.9206 | Val Loss: 0.2091 | Val Acc: 0.9259 | Val Macro-F1: 0.9271
Epoch 05/15 | Train Loss: 0.1883 | Train Acc: 0.9464 | Val Loss: 0.1515 | Val Acc: 0.9537 | Val Macro-F1: 0.9549
Epoch 06/15 | Train Loss: 0.1081 | Train Acc: 0.9802 | Val Loss: 0.1390 | Val Acc: 0.9537 | Val Macro-F1: 0.9549
Epoch 07/15 | Train Loss: 0.1159 | Train Acc: 0.9683 | Val Loss: 0.1187 | Val Acc: 0.9630 | Val Macro-F1: 0.9640
Epoch 08/15 | Train Loss: 0.0677 | Train Acc: 0.9901 | Val Loss: 0.1099 | Val Acc: 0.9722 | Val Macro-F1: 0.9730
Epoch 09/15 | Train Loss: 0.0744 | Train Acc: 0.9861

In [ ]:
# ============================================================
# SAVE EFFICIENTNET-B0 RESULTS
# ============================================================

from pathlib import Path
import pandas as pd
import torch

RESULTS_DIR = Path(
    "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final/"
    "Stage2A_Results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save training history


history_df = pd.DataFrame(history)

history_file = (
    RESULTS_DIR /
    "efficientnet_b0_history.csv"
)

history_df.to_csv(
    history_file,
    index=False
)


# Save best model


model_file = (
    RESULTS_DIR /
    "efficientnet_b0_best.pth"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "classes": classes,
        "class_to_idx": class_to_idx,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1,
        "seed": SEED,
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-4
    },
    model_file
)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------


print("EFFICIENTNET-B0 SAVED")


print("Model exists:", model_file.exists())
print("History exists:", history_file.exists())

print("Model:", model_file)
print("History:", history_file)

print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_val_f1)

EFFICIENTNET-B0 SAVED
Model exists: True
History exists: True
Model: /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage2A_Results/efficientnet_b0_best.pth
History: /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage2A_Results/efficientnet_b0_history.csv
Best epoch: 13
Best validation Macro-F1: 0.981981981981982
